# Data ingestion layer with PII masking
welcome! this is a data pipeline I am creating to practice and reinforce concepts in Data Engineering using data from a local business. The application will run entirely on my machine.

# The data pipeline will flow as follows: 
## Data is ingested in batches daily, from multiple locations. 
- The goal of the Data Ingestion process: 
    -   perform a **Full Load** of the current database, 
    -   then persist with any CDC (**Change Data Capture**) from that moment forward.

## Objectives:
- **Foundations**: Explore Database Schema and Tables to determine initial partition and bucket structure.
- **Batch Orchestration**: After Full Load, create other notebook to run on daily basis (batch process) that will update the database to capture all new transactions from all locations daily. 
- **Security**: Identify all PII columns, then mask or encrypt (using hashlib) those columns prior to storing in bronze layer tables.
- **Storage**: Store the masked data in a separate folder called "Masked Data_bronze", with one Folder for each Day (partition) containing one file for each location for that day (bucket).
- **Best Practices**: Use a separate code notebook to execute further transformations, as we want to emphasize that once the data is masked via this process, it will not be dangerous to expose to consumers of the bronze layer of this database.

In [ ]:
import pandas as pd
import hashlib
import json
import os
from faker import Faker
from faker_vehicle import VehicleProvider

In [ ]:
# enforcing string dtypes on columns with PII for encryption
raw_txns = pd.read_csv('Raw Data/2026_sept_txns_raw.csv',
                       dtype={'#': 'str', 'Ref No': 'str', 'Reservation ID': 'str', 
                              'User/Preparer': 'str', 'Client Name': 'str', 'Client Fee Override Approved By': 'str',
                                'Client Rate Override Approved By': 'str', 'Approved By': 'str', 'E-mail': 'str', 
                                'Phone': 'str', 'ID Number': 'str', 'Source of Funds': 'str', 
                                'Occupation': 'str', 'Notes': 'str', 'ID Number': 'str'})
col_count = len(raw_txns.columns)
print(raw_txns.columns)
raw_txns.isna().sum().nlargest(10).sort_values(ascending=False)

In [ ]:
raw_txns.dtypes

In [ ]:
raw_txns.head()

### Findings: 
- We can delete the "Total" row from the raw input file and the empty row above it (row #s 765 and 766). These two rows do not have a Ref No. 
- Ref No is not unique. Are they duplicated in the event that multiple currencies were sold or purchased as part of that transaction? Address in Silver layer.

In [ ]:
raw_txns = raw_txns.dropna(subset=['Ref No'])
raw_txns.head()

#### We have 64 columns in the raw files from the transactions database containing all transactions in September for multiple locations. 
0. Confirm index.
1. Create Date column to partition by.
2. Standardize Location column for Bucket naming.
3. Mask or Encrypt PII columns, storing a key to a new folder call secrets_map that contains a mapping of the results of the masking or encryption:
    - Company - Location Faker (seeded instance)
    - User/Preparer - Faker (seeded instance)
    - Account Holder - Faker (seeded instance)
    - Client Name - Faker (seeded instance)
    - Client Fee Override Approved By - Faker (seeded instance)
    - Client Rate Override Approved By - Faker (seeded instance)
    - Company Name - Faker (seeded instance)
    - E-mail - Faker (seeded instance)
    - Approved By - Faker (seeded instance)
    - Phone - Faker (seeded instance)
    - ID Number - hashlib
4. Store the masked/encyrpted database into separate files based on the partition (date) and bucket (location) keys. 

In [ ]:
# confirmed that Ref No will be our transaction index. 
# lets rename it 
raw_txns = raw_txns.rename(columns={'Ref No': 'Transaction ID'})
raw_txns.head()


In [ ]:
# to partition the data as we ingest it into storage, we want to isolate the files by Date
# we can create a column with dtype of date for partitioning
raw_txns['Date'] = pd.to_datetime(raw_txns['Date and Time Entered'], format='mixed').dt.date
raw_txns[['Date and Time Entered', 'Date']].head(5)

In [ ]:
# evaluate the location data to see unique values
raw_txns['Company - Location'].unique()

In [ ]:
# reformat 'Company - Location' to remove the Company name that precedes the location, which we also may want to keep private.
raw_txns['Location'] = raw_txns['Company - Location'].str[27:]
raw_txns[['Transaction ID', 'Location', 'Company - Location', 'Currency Code', 'Type']].head(5)

In [ ]:
# remove company for added security and privacy
raw_txns = raw_txns.drop(['Company - Location'], axis=1)

In [ ]:
#confirmed that we have a unique identifier for each transaction, needed when a transaction involves multiple currencies.
raw_txns['uid'] = raw_txns['Transaction ID'].astype(str) + '_' + raw_txns['Currency Code'].astype(str)
raw_txns = raw_txns.set_index(['uid'])
raw_txns[['Transaction ID', 'Currency Code', 'Location']].head(5)

#### We have completed the following tasks. 
##### We can now partition by date and bucket by location during the full load into the storage directory after we mask the sensitive data. 
##### We have also enforced data quality best practices by denying records with null "Ref No"/Transaction ID's into our database, given the structure of the source file. 
##### Our current index is a concatenation of the Transaction ID and Currency Code to guarantee uniqueness according to the source database configuration.
0. ~~Confirm index.~~
1. ~~Create Date column to partition by.~~
2. ~~Standardize Location column for Bucket naming.~~

##### As part of overall data privacy concerns, we want to delete unnecesarry columns. Additionally, on platforms like AWS S3, there are cheaper storage tiers were archival or compliance-only data can be stored. In this case, we will focus on masking. 
 if we were on AWS or another platform like Databricks, we could implement policies that protect the assets in our bronze layer with the appropriate credentials and authentication as needed, for example in the case of compliance audits. Otherwise, it can be masked for use by general users. 

### Now move on to step 3: masking and encryption

3. Mask or Encrypt PII columns, storing a key to a new folder call secrets_map that contains a mapping of the results of the masking or encryption:
    - Location - Faker (seeded instance)
    - User/Preparer - Faker (seeded instance)
    - Account Holder - Faker (seeded instance)
    - Client Name - Faker (seeded instance)
    - Client Fee Override Approved By - Faker (seeded instance)
    - Client Rate Override Approved By - Faker (seeded instance)
    - Company Name - Faker (seeded instance)
    - E-mail - Faker (seeded instance)
    - Approved By - Faker (seeded instance)
    - Phone - Faker (seeded instance)
    - ID Number - hashlib
    - Source of Funds
    - Occupation
    - Notes

In [ ]:
#all the above fields that we want to make or encrypt should be normalized (strip whitespace, convert to lowercase, etc.), 
#so that the masking function can have a consistent input format.

# initiate use of Faker to mask the data deterministically
fake = Faker()

#driver's license ID's will become License Plate numbers
fake.add_provider(VehicleProvider)

# this will be the data we store to the secrets_map file in case of need for decrypting in case of actual fraud detection
location_mapping = {}
name_mapping = {}
company_mapping = {}
email_mapping = {}
phone_mapping = {}
id_number_mapping = {}
fund_source_mapping = {}
occupation_mapping = {}
notes_mapping = {}

#list of mapping for use in case of fraud detection
secrets_map = {'loc': location_mapping, 'name': name_mapping, 'company': company_mapping, 'email': email_mapping, 'phone': phone_mapping, 
               'id_number': id_number_mapping, 'fund_source': fund_source_mapping, 'occupation': occupation_mapping, 'notes': notes_mapping}

#helper functions to generate masked data with deterministic seeding while storing mapping of the original values
def get_fake_location(i):
    fake.seed_instance(i)
    a = fake.city()
    location_mapping[i] = a
    return a

def get_fake_person_name(i):
    fake.seed_instance(i)
    a = fake.name()
    name_mapping[i] = a
    return a

def get_fake_company_name(i):
    fake.seed_instance(i)
    a = fake.company()
    company_mapping[i] = a
    return a

def get_fake_email(i):
    fake.seed_instance(i)
    a = fake.email()
    email_mapping[i] = a
    return a

def get_fake_phone(i):
    fake.seed_instance(i)
    a = fake.phone_number()
    phone_mapping[i] = a
    return a

def get_fake_id_number(i):
    fake.seed_instance(i)
    a = fake.license_plate()
    id_number_mapping[i] = a
    return a

def get_fake_fund_source(i):
    fake.seed_instance(i)
    a = fake.bs()
    fund_source_mapping[i] = a
    return a

def get_fake_job_title(i):
    fake.seed_instance(i)
    a = fake.job()
    occupation_mapping[i] = a
    return a

def get_fake_notes(i):
    fake.seed_instance(i)
    a = fake.sentence()
    notes_mapping[i] = a
    return a


In [ ]:
# iterate over the list of columns to mask and generate our masked dataset
# we will use faker for all columns except ID Number, which will use hashlib
#List of PII columns to iterate over
cols_to_mask = [
    'Location',
    'User/Preparer',
    'Client Name',
    'Client Fee Override Approved By',
    'Client Rate Override Approved By',
    'Company Name',
    'E-mail',
    'Approved By',
    'Phone',
    'ID Number',
    'Source of Funds',
    'Occupation',
    'Notes' 
]

def mask_pii_data(raw_df: pd.DataFrame, PII_cols: list):

    masked_df = raw_df.copy()

    for x in PII_cols:
        if x == 'Location':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_location(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x in ['User/Preparer', 'Client Name', 'Client Fee Override Approved By', 'Client Rate Override Approved By', 'Approved By']:
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_person_name(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'Company Name':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_company_name(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'E-mail':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_email(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'Phone':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_phone(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'ID Number':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            # use hashlib for ID Number
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: hashlib.sha256(y.encode('utf-8')).hexdigest() if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'Source of Funds':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_fund_source(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'Occupation':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_job_title(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

        elif x == 'Notes':
            # important that we convert all strings to lowercase and remove extra whitespace prior to calling helper function for consistency
            tmp = masked_df[x].str.lower().str.replace('\s+', '', regex=True)
            masked_df[x] = tmp.apply(lambda y: get_fake_notes(y) if (pd.notnull(y) and y != '' and str(y) != '0') else y)

    return masked_df 


In [ ]:
masked_txns = mask_pii_data(raw_txns, cols_to_mask)

In [ ]:
print(fund_source_mapping)

In [ ]:
masked_txns.head()

In [ ]:
### To test step 3 of the ingestion pipeline, we will need to verify that we have masked all PII columns correctly and stored the mapping information. 
# First write the masked data to a new CSV file for testing

# masked_txns.to_csv('test_bronze/2026_sept_txns_masked.csv', index=False)

### Testing showed that source database has PII entered in a unexpected fields including "Source of Funds", "Occupation", and potentially other user-entered fields such as "Notes". I have gone back and added those to cols_to_mask for masking while preserving their mapping in case of pattern recognition or compliance needs.

##### We can now store the masking key to secret_maps.txt to preserve traceability of the masking process

In [ ]:
def write_secrets_map(folder: str):
    for i in secrets_map.keys():
        with open(folder + "/" + str(i) + ".json", "w") as f:
            json.dump(secrets_map[i], f, indent=4, sort_keys=True)

write_secrets_map('encryption')

### We have now completed task 3 and can move on to the final step of our initial ingestion into the bronze layer of our pipeline.

3. Mask or Encrypt PII columns, storing a key to a ~~new folder call secrets_map~~ that contains a mapping of the results of the masking or encryption:
    - ~~Location - Faker (seeded instance)~~
    - ~~User/Preparer - Faker (seeded instance)~~
    - ~~Account Holder - Faker (seeded instance)~~
    - ~~Client Name - Faker (seeded instance)~~
    - ~~Client Fee Override Approved By - Faker (seeded instance)~~
    - ~~Client Rate Override Approved By - Faker (seeded instance)~~
    - ~~Company Name - Faker (seeded instance)~~
    - ~~E-mail - Faker (seeded instance)~~
    - ~~Approved By - Faker (seeded instance)~~
    - ~~Phone - Faker (seeded instance)~~
    - ~~ID Number - hashlib~~
    - ~~Source of Funds~~
    - ~~Occupation~~
    - ~~Notes~~

#### In this case our secrets_map folder is called 'encryption' and contains a separate json file for each PII column with a mapping.

#### We can now move on to the final stage of this notebook, saving the data to a database partitioned by date and bucketed by location

4. Store the masked/encyrpted database into separate files based on the partition (date) and bucket (location) keys. 

In [ ]:
#set destination folder for the partitioned and bucketed data
bronze_layer = "Masked Data_bronze/"

masked_txns.groupby(['Date', 'Location']).first()

In [ ]:
for (date, loc), group in masked_txns.groupby(['Date', 'Location']):
    if not os.path.exists(bronze_layer + str(date) + '/' + str(loc)):
        os.makedirs(bronze_layer + str(date) + '/' + str(loc))
        group.to_csv(bronze_layer + str(date) + '/' + str(loc) + '/' + str(date) + '_' + str(loc) + '_' + 'data.csv', index=True) #index must be included for uniqueness

### Our ingestion layer into the Bronze table is now complete. We have now completed all tasks:

0. ~~Confirm index.~~
1. ~~Create Date column to partition by.~~
2. ~~Standardize Location column for Bucket naming.~~
3. ~~Mask or Encrypt PII columns, storing a key to a new folder call secrets_map that contains a mapping of the results of the masking or encryption:~~
   - ~~Location - Faker (seeded instance)~~
    - ~~User/Preparer - Faker (seeded instance)~~
    - ~~Account Holder - Faker (seeded instance)~~
    - ~~Client Name - Faker (seeded instance)~~
    - ~~Client Fee Override Approved By - Faker (seeded instance)~~
    - ~~Client Rate Override Approved By - Faker (seeded instance)~~
    - ~~Company Name - Faker (seeded instance)~~
    - ~~E-mail - Faker (seeded instance)~~
    - ~~Approved By - Faker (seeded instance)~~
    - ~~Phone - Faker (seeded instance)~~
    - ~~ID Number - hashlib~~
    - ~~Source of Funds~~
    - ~~Occupation~~
    - ~~Notes~~
4. ~~Store the masked/encrypted database into separate files based on the partition (date) and bucket (location) keys.~~